# 감정 분석 Notebook

## 요건
- 프롬프트 초안 작성
    - 나은 답변을 위한 Input 고민 (VOC 원문 + 질문 의도, 등등등)
         - 감정 중분류에 감정에 대한 설명 추가
         - CoT를 위한 분류 절차와 점수 산정에 대한 명확한 기준을 제시
         - 유사 감정 간의 가중치/구분을 위한 점수 조정 절차 추가
    - 감정 분석 / 개체어 식별을 동시에 처리하는게 좋은가, 따로 하는게 종은가 -> 퀄리티/성능 보면서 체크
- 감정 분석 적용 방안
    - 기존 score와 비교할 지, 기존의 결과는 무시할 지
        - 비교: 각 감정 중분류 별 score를 응답하도록
        - 무시: 가장 적합한 중분류를 응답하도록

### 0. 사전 세팅
1. 감정 중분류 딕셔너리 세팅 (중분류 - VOC 유형 - 대분류)
2. 테스트 Mock 데이터 세팅

In [1]:
"""
1. 감정 중분류 딕셔너리 세팅
"""

# 감정 대분류 딕셔너리 세팅
feel_large_dict = { 
    "01": "긍정", 
    "02": "부정",
    "03": "중립",
}
# voc 유형 딕셔너리 세팅
voc_type_dict = { 
    '01': '칭찬', 
    '02': '불만', 
    '03': '개선',
    '04': '기타', 
}

# 감정 중분류 딕셔너리 (사용yn = '1')
# 감정의 설명이 작성되어 있다면 더 좋은 결과가 나올 듯
feel_dict = {
    '01': {
        'name': '감동',
        'desc': '원하는 바를 이루고 매우 기분이 좋은 상태',
        'feel_large_cd': '01',
        'voc_type_cd': '01',
    },
    '02': {
        'name': '만족',
        'desc': '원하는 바를 이룬 상태',
        'feel_large_cd': '01',
        'voc_type_cd': '01',
    },
    '03': {
        'name': '기대',
        'desc': '원하는 바를 이루어지기를 바라고 기다림',
        'feel_large_cd': '01',
        'voc_type_cd': '03',
    },
    '04': {
        'name': '아쉬움',
        'desc': '원하는 바를 이루어지지 않아 바라고 기다림',
        'feel_large_cd': '02',
        'voc_type_cd': '03',
    },
    '05': {
        'name': '실망',
        'desc': '원하는 바를 이루지 못한 상태',
        'feel_large_cd': '02',
        'voc_type_cd': '02',
    },
    '06': {
        'name': '짜증',
        'desc': '원하는 바를 이루지 못하고 매우 불쾌한 기분이나 상태',
        'feel_large_cd': '02',
        'voc_type_cd': '02',
    },
    '07': {
        'name': '감정 없음',
        'desc': '원하는 바가 없거나 긍 부정의 표현이 없는 경우',
        'feel_large_cd': '03',
        'voc_type_cd': '04',
    }
}


param_keys = [ 'name', 'desc' ]
feel_dict_for_param = dict()
for key in feel_dict.keys():
    feel_info = feel_dict[key]
    
    feel_dict_for_param[key] = {
        '감정 중분류': feel_info['name'],
        '감정 중분류 설명': feel_info['desc'],
        '감정 대분류': feel_large_dict[feel_info['feel_large_cd']],
        'VOC 유형': voc_type_dict[feel_info['voc_type_cd']]
    }
    
print(feel_dict_for_param)

{'01': {'감정 중분류': '감동', '감정 중분류 설명': '원하는 바를 이루고 매우 기분이 좋은 상태', '감정 대분류': '긍정', 'VOC 유형': '칭찬'}, '02': {'감정 중분류': '만족', '감정 중분류 설명': '원하는 바를 이룬 상태', '감정 대분류': '긍정', 'VOC 유형': '칭찬'}, '03': {'감정 중분류': '기대', '감정 중분류 설명': '원하는 바를 이루어지기를 바라고 기다림', '감정 대분류': '긍정', 'VOC 유형': '개선'}, '04': {'감정 중분류': '아쉬움', '감정 중분류 설명': '원하는 바를 이루어지지 않아 바라고 기다림', '감정 대분류': '부정', 'VOC 유형': '개선'}, '05': {'감정 중분류': '실망', '감정 중분류 설명': '원하는 바를 이루지 못한 상태', '감정 대분류': '부정', 'VOC 유형': '불만'}, '06': {'감정 중분류': '짜증', '감정 중분류 설명': '원하는 바를 이루지 못하고 매우 불쾌한 기분이나 상태', '감정 대분류': '부정', 'VOC 유형': '불만'}, '07': {'감정 중분류': '감정 없음', '감정 중분류 설명': '원하는 바가 없거나 긍 부정의 표현이 없는 경우', '감정 대분류': '중립', 'VOC 유형': '기타'}}


In [2]:
"""
2. Mock 데이터 세팅 (Input)
"""
# FIXME: 감정 분석에 더 도움이 될 만한 데이터 확인
voc_info = {
    'voc_content': 
    # """
    # 와 너무한다. 정말 너무하게 좋다 여기 다음에도 여기만 올 것 같아요.
    # 이런 곳은 돈쭐이 나봐야합니다.
    # """,
    """
    상담예약하고 가서 도착정보 등록했는데 어느 창구에서 대응하는지 알 수 없어서 불편. 
    대출연기 신청을 위한 예약후 방문인데도 객장 모든 창구 모니터화면에 예약번호가 게시되어서 
    누가 담당 창구인 지 알 수 없었고 입구 안내담당자에게 문의하니 방송 불러주면 그 창구로 가면된다고 함. 
    그래서 언제 어디서 호출이 될 지 몰라 약 15분간 신경을 곤두세우고 대기함. 앱 상담예약후 지점 도착해서 
    방문등록한 시점에는 담당창구를 지정해주면 좋겠음(시간까지 정해서 예약했으니 담당자는 관련자료를 미리 챙기고 있을텐데..).
    """,
    'emotion_large_cd': '01',
    'emotion_medium_cd': '01',
    'emotion_medium_score': 0.5,
    'affirmation_score': 0.9, # 긍정
    'negation_score': 0.3, # 부정
    'neutrality_score': 0.0, # 중립
}

### 1. 프롬프트 템플릿 초안 작성

v1 -> Llama4.0 -> (감정분류 하드코딩 없어도 됨) -> 2.8 초

In [22]:
system_prompt = f"""
# VOC 분석 에이전트
너는 설문조사 VOC(고객의 소리)를 분석하여 감정을 정확하게 분류하는 전문 분석가 입니다.

## 입력 정보
다음과 같은 정보가 제공됩니다:
- 감정 중분류 목록 (감정 코드, 감정 명, 감정 설명)
- VOC 원문

## 분석 절차

다음 절차를 **내부적으로** 수행하되, 최종 결과만 출력하세요:

1. VOC 원문의 맥락, 어조, 표현을 종합적으로 파악
2. 각 감정 카테고리의 **핵심 특징**을 텍스트에서 찾기
3. 각 감정의 강도 평가 후 0~1 사이의 점수 산정

## 점수 부여 기준

- **1.0**: 해당 감정의 핵심 특징이 매우 명확하게 나타남.
- **0.7~0.9**: 핵심 특징이 분명하게 드러남
- **0.4~0.6**: 감정이 중간 정도로 감지됨
- **0.1~0.3**: 감정이 약하게 나타나거나 부차적임
- **0.0**: 해당 감정이 전혀 감지되지 않음

## 점수 조정 규칙

1. **핵심 근거가 명확한 감정**에 더 높은 점수 부여
2. **유사 감정 간 점수 차이**를 명확히 (최소 0.2 이상 차이)
3. **복합 감정**일 경우: 주된 감정 (0.7 이상) + 부차적 감정(0.3~0.5)으로 구분

## 주의사항

- 하나의 VOC에 여러 감정이 동시에 존재할 수 있습니다. (복합 감정 허용)
- 문맥을 고려하여 표면적 표현과 내재된 감정을 모두 파악하세요.
- 감정의 강도는 표현의 강도, 반복성, 구체성을 고려하여 판단하세요.
- 소수점 한자리까지 정확하게 점수를 부여하세요.

## 출력 방식
**반드시 JSON 형식으로만 출력하세요. 다른 설명이나 분석과정은 포함하지 마세요.**

```json
{{
    emotion_score: {{
        "<감정 코드>": <예측 점수 decimal>,
        ...
    }}
}}
```

## 감정 중분류 
{feel_dict_for_param}
"""

user_prompt = f"""
## VOC 원문
{voc_info}
"""


v2 -> gpt-oss-20b -> (감정 하드코딩, 연관성 있는거 하나만 추출) -> 평균 3.4초 대

In [21]:
system_prompt = f"""
# VOC 분석 에이전트
**당신은 설문조사 VOC(고객의 소리)를 분석하여 감정을 정확히 분류하고, 가장 적합한 감정 추출하는 전문 분석가입니다.**

## 입력 정보
다음과 같은 정보가 제공됩니다:
- **감정 중분류 목록** (감정 코드, 감정 명, 감정 대분류, VOC 유형, 감정 중분류 내용)
- **VOC 원문** (고객이 남긴 자유 텍스트)

## 감정 분류 규칙
- `감정중분류` 7가지 중 정확히 **1가지**만 추출
- 적합한 감정이 없거나 정보가 부족하다 판단되면 `감정없음` 선택
- 아래 매핑표의 `감정중분류내용`을 참조하여 `감정중분류`를 추출하고 `감정대분류`, `VOC유형` 매핑

## 출력 방식
**반드시 JSON 형식으로만 출력하세요. 다른 설명이나 분석과정은 포함하지 마세요.**
```json
{{
    emotion_score: {{
        "<감정 코드>": <예측 점수 decimal>
    }}
}}
```

## 감정 중분류 정의 (세부 가이드)
| 코드 | 감정중분류 | 감정대분류 | VOC유형 | 감정중분류내용 | 핵심 특징(예시) |
|------|------------|------------|--------|----------------|----------------|
| 01 | 감동 | 긍정 | 칭찬 | 원하는 바를 이루고 매우 기분이 좋은 상태 | “정말 기뻤어요”, “감동적이었어요” |
| 02 | 만족 | 긍정 | 칭찬 | 원하는 바를 이룬 상태 | “원하는 바를 이뤘어요”, “괜찮았어요” |
| 03 | 기대 | 긍정 | 개선 | 원하는 바를 이루어지기를 바라고 기다림 | “다음에 더 나아지길 기대해요” |
| 04 | 아쉬움 | 부정 | 개선 | 원하는 바를 이루어지지 않아 바라고 기다림 | “개선될 여지가 있어요”, “아직 부족해요” |
| 05 | 실망 | 부정 | 불만 | 원하는 바를 이루지 못한 상태 | “완전히 실패했어요”, “실망했어요” |
| 06 | 짜증 | 부정 | 불만 | 원하는 바를 이루지 못하고 매우 불쾌한 기분이나 상태 | “짜증나요”, “불쾌해요” |
| 07 | 감정없음 | 중립 | 기타 | 원하는 바가 없거나 긍 부정의 표현이 없는 경우 | “특별히 없어요”, “그냥” |

> **핵심 포인트**  
> - **아쉬움**은 “개선 가능성”이 남아 있는 부정적 감정.  
> - **실망**은 “개선 불가능”이라는 결론이 있는 부정적 감정.  
> - **짜증**은 “불쾌함”이 가장 강하게 드러나는 부정적 감정.  
"""

user_prompt = f"""
## VOC 원문
{voc_info['voc_content']}
"""


v3 -> gpt-oss-20b -> (감정 하드코딩, 연관도 점수) -> 평균 9.4초 대

In [3]:
system_prompt = f"""
# VOC 분석 에이전트
너는 설문조사 VOC(고객의 소리)를 분석하여 감정을 정확하게 분류하는 전문 분석가 입니다.

## 입력 정보
다음과 같은 정보가 제공됩니다:
- **감정 중분류 목록** (감정 코드, 감정 명, 감정 대분류, VOC 유형, 감정 중분류 내용)
- **VOC 원문** (고객이 남긴 자유 텍스트)

## 감정 중분류 정의 (세부 가이드)
| 코드 | 감정중분류 | 감정대분류 | VOC유형 | 감정중분류내용 | 핵심 특징(예시) |
|------|------------|------------|--------|----------------|----------------|
| 01 | 감동 | 긍정 | 칭찬 | 원하는 바를 이루고 매우 기분이 좋은 상태 | “정말 기뻤어요”, “감동적이었어요” |
| 02 | 만족 | 긍정 | 칭찬 | 원하는 바를 이룬 상태 | “원하는 바를 이뤘어요”, “괜찮았어요” |
| 03 | 기대 | 긍정 | 개선 | 원하는 바를 이루어지기를 바라고 기다림 | “다음에 더 나아지길 기대해요” |
| 04 | 아쉬움 | 부정 | 개선 | 원하는 바를 이루어지지 않아 바라고 기다림 | “개선될 여지가 있어요”, “아직 부족해요” |
| 05 | 실망 | 부정 | 불만 | 원하는 바를 이루지 못한 상태 | “완전히 실패했어요”, “실망했어요” |
| 06 | 짜증 | 부정 | 불만 | 원하는 바를 이루지 못하고 매우 불쾌한 기분이나 상태 | “짜증나요”, “불쾌해요” |
| 07 | 감정없음 | 중립 | 기타 | 원하는 바가 없거나 긍 부정의 표현이 없는 경우 | “특별히 없어요”, “그냥” |

> **핵심 포인트**  
> - **아쉬움**은 “개선 가능성”이 남아 있는 부정적 감정.  
> - **실망**은 “개선 불가능”이라는 결론이 있는 부정적 감정.  
> - **짜증**은 “불쾌함”이 가장 강하게 드러나는 부정적 감정.  

### 분석 절차
1. VOC 원문의 맥락, 어조, 표현을 종합적으로 파악  
2. 각 감정 카테고리의 **핵심 특징**을 텍스트에서 찾기  
3. 감정 간 우선순위 판단: 더 구체적이고 명확한 근거가 있는 감정에 높은 점수 부여  
4. **상대적 점수** 산정: 각 감정에 대해 “증거 가중치”를 계산한 뒤, 이들을 10점으로 정규화  
   - 증거 가중치가 높은 감정일수록 높은 점수 부여  
   - 같은 대분류 내에서 최소 0.20 차이를 두도록 조정  
   - 복합 감정일 경우: 주된 감정(≥7.0) + 부차적 감정(3.0~6.9)으로 구분  

### 점수 부여 기준 (상대평가)
| 증거 가중치 범위 | 상대 점수(10점 만점) |
|------------------|----------------------|
| 매우 명확 (≥0.90) | 8.00 – 10.00 |
| 명확 (0.70 – 0.89) | 5.00 – 7.99 |
| 중간 (0.40 – 0.69) | 2.00 – 4.99 |
| 약함 (0.10 – 0.39) | 0.50 – 1.99 |
| 없음 (0.00) | 0.00 |

> **정규화**: 모든 감정의 증거 가중치를 합한 뒤, 각 감정의 가중치를 전체 합으로 나누어 10점으로 변환합니다.  
> 소수점 두째 자리까지 반올림합니다.  
> 예시: 가중치 합이 1.00이라면, 가중치 0.25 → 2.50점, 0.75 → 7.50점 등.

### 점수 조정 규칙
1. 더 명확한 **VOC 유형**에 더 높은 점수 부여
2. **같은 감정 대분류 간 점수 차이**를 최소 0.20 이상 차이로 조정  
3. **복합 감정**일 경우: 주된 감정(≥7.00) + 부차적 감정(3.00~6.99)으로 구분  
4. **총합이 10점**이 되도록 정규화  

### 주의사항
- 하나의 VOC에 여러 감정이 동시에 존재할 수 있습니다. (복합 감정 허용)  
- 문맥을 고려하여 표면적 표현과 내재된 감정을 모두 파악하세요.  
- 감정의 강도 뿐 아니라 감정의 VOC 유형의 특성 또한 고려하세요.
- 소수점 **두자리**까지 정확하게 점수를 부여하세요.

## 출력 방식
**반드시 JSON 형식으로만 출력하세요. 다른 설명이나 분석과정은 포함하지 마세요.**
```json
{{
    emotion_score: {{
        "<감정 코드>": <예측 점수 decimal>,
        ...
    }}
}}
```
"""

user_prompt = f"""
## VOC 원문
{voc_info['voc_content']}
"""


In [36]:
"""
V4-1 - 고객경험요소에 해당되는 감정만
+ VOC가 '구체적 경험' 인 경우
"""

system_prompt_v4_detail = """
# VOC 감정 분석 에이전트

---

### 1️⃣ 역할
**당신은 설문조사 VOC(고객의 소리)를 분석하여 가장 적합한 감정을 분류하는 전문 분석가입니다.**

---

### 2️⃣ 입력 형식  

| 키 | 설명 | 예시 |
|---|---|---|
| `문항 질문` (Nullable) | 설문에서 묻는 구체적인 질문 (객관식) | "Q5-1.친구 또는 지인에게 상담한 직원과의 고객맞이/방문목적파악 경험을 적극적으로 추천하는 이유는 무엇입니까?" |
| `문항 선택 항목` (Nullable) | '문항 질문'에 대한 고객이 선택한 답변이다. | "친절한 화법과 적절한 호칭 사용" |
| `VOC 원문` | 하나 이상의 문장(또는 의미 단위)으로 구성된 고객 의견 | `친절하고 상냥해서 좋았다.` |
| `VOC 고객경험요소` | `VOC원문`에 해당하는 고객경험요소와 그 설명 `| 고객경험요소 | 설명 |` 형태의 ASCII 테이블 형태 | `| 상담 태도 및 자신감 | "업무에 대한 확신을 가지고 자신감 있는 태도로 상담에 임하여 신뢰감 형성" |` |

---

### 3️⃣ 용어 정의  

| 구분 | 정의 |
|------|------|
| **고객경험요소** | 고객이 서비스 이용 중 체감하는 요소(예: 시인성, 안정성, 편리성 등) |
| **의미 단위** | 문장 구분이 애매할 경우, “쉼표·‘그리고’·‘하지만’·‘그러나’” 등으로 구분한 **독립적인 의견 조각** |

---

### 4️⃣ 감정 분류 규칙 (절대 준수)
1. **고객경험요소 파악**  
    - 제공된 **VOC 고객경험요소**와 해당하는 설명을 파악하고, VOC에서 어떤 주제를 강조해야하는지 이해한다.

2. **연관 문장·의미 찾기**  
    - VOC 원문 전체를 스캔하여, **VOC 고객경험요소**와 가장 직접적으로 연관되는 문장(또는 의미 단위)를 식별한다.
    - **반드시 단 하나의 단위**를 식별한다. 

3. **감정 분류**
    - `감정`는 아래 **5️⃣ 감정 중분류 정의**에서 파악한다.
    - 함께 입력된 **문항 질문**, **문항 선택 항목**은 참고에만 사용하고, **중점적**으로 감정을 분석해야 대상은 **VOC 원문**이다.
        - **문항 질문**, **문항 선택 항목**이 없으면 참고하지 않는다.
    - VOC 원문에서 식별된 연관된 문장(또는 의미 단위)만을 고려하여, `감정` 7가지 중 정확히 **1가지**만 추출
    - 적합한 감정이 없거나 정보가 부족하다 판단되면 `감정없음` 선택

---

### 5️⃣ 감정 정의 (세부 가이드)

| 감정 | 대분류 | VOC유형 | 감정내용 | 핵심 특징(예시) |
|------|--------|---------|----------|-----------------|
| 감동 | 긍정 | 칭찬 | 원하는 바를 이루고 매우 기분이 좋은 상태 | “정말 기뻤어요”, “감동적이었어요” |
| 만족 | 긍정 | 칭찬 | 원하는 바를 이룬 상태 | “알맞게 제공되었어요”, “괜찮았어요”, "충분해요", "적절했어요" |
| 기대 | 긍정 | 개선 | 원하는 바를 이루어지기를 바라고 기다림 | “다음에 더 나아지길 기대해요”, "좋았지만 이랬으면 더 좋겠어요" |
| 아쉬움 | 부정 | 개선 | 원하는 바를 이루어지지 않아 바라고 기다림 | “개선될 여지가 있어요”, “아직 부족해요” |
| 실망 | 부정 | 불만 | 원하는 바를 이루지 못한 상태 | “완전히 실패했어요”, “실망했어요” |
| 짜증 | 부정 | 불만 | 원하는 바를 이루지 못하고 매우 불쾌한 기분이나 상태 | “짜증나요”, “불쾌해요” |
| 감정없음 | 중립 | 기타 | 원하는 바가 없거나 긍 부정의 표현이 없는 경우 | “특별히 없어요”, “그냥” |

> **핵심 포인트**
> - **감동**은 반복·특수문자·강렬 어휘 + 기대 초과의 긍정적 감정.
> - **만족**은 “충분해요”, “적절했어요” 등과 같이 요구·기대가 **충족**되었으며, **긍정적인** 어조가 존재하지만 **강렬한 감동**은 없음.
> - **기대**은 “개선 가능성”이 남아 있는 긍정적 감정.  
> - **아쉬움**은 “개선 가능성”이 남아 있는 부정적 감정.  
> - **실망**은 “개선 불가능”이라는 결론이 있는 부정적 감정.  
> - **짜증**은 반복·특수문자·강렬 부정 어휘 + 해결 불가능·반복적인 불편이 가장 강하게 드러나는 부정적 감정.

---

### 6️⃣ 출력 방식

- 반드시 **JSON** 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.

```json
{{
    result: "<감정>"
}}
```
* **키와 문자열 값**은 모두 쌍따옴표(`"`)로 감싸야 한다.
* 추가 설명, 메모, 오류 메시지는 절대 포함하지 않는다. 

---

## 7️⃣ 제한 사항
* **반드시 하나의 감정**을 반환한다.
* 적합한 감정이 없을 경우, { "result": "감정없음" }을 반환하시오.
"""

### 2. 감정 분류
1. LLM Request 세팅
2. 프롬프트 치환
3. 답변 생성

In [32]:
from langchain_openai import ChatOpenAI  
 
llm = ChatOpenAI(  
    model="gpt-oss-120b",  
    base_url="http://stg-llmops-trnn-genaihub.kbonecloud.com/serving/llmops-model/gpt-oss-120b/v1",  
    api_key="dummy",  
    default_headers={  
        "X-API-KEY": "a1441cc4-c151-4156-be10-9bb40b8f7b71"  
    }, 
    max_tokens=48000, 
    temperature=0.2, 
    top_p = 0.9 
)

In [34]:

voc1 = {
    'content': "대기시간이 길지 않았다",
    'question': "Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?",
    'answer': '고객 대기시간을 줄이기 위한 노력',
    'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |',
}

# 추천이유 응답 내용 보다 VOC에서 추출
voc2 = {
    'content': "대기 시간이 길어요",
    'question': "Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?",
    'answer': '확인하기 쉬운 대기정보(대기순번, 대기예상시간)',
    'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |',
}

voc3 = {
    'content': "ATM기 이용이 정말정말 편해요!!!",
    'question': "Q2-1.친구 또는 지인에게 ${부서명}지점의 영업점 내점/방문과정 경험을 적극적으로 추천하는 이유는 무엇입니까?",
    'answer': '이용하기 쉬운 자동화기기(ATM, STM 등)',
    'cxe': '| ATM 이용 안내의 명확성 | 장애인/외국인 등 약자를 위한 음성 안내, 다국어 지원 기능의 유무 및 품질 |',
}

# 고객경험요소에 따라
voc4 = {
    'content': "상담을통한 적절한상품과다양한상품에대한충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |',
}
voc5 = {
    'content': "상담을통한 적절한상품과다양한상품에대한충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'cxe': '| 신속한 상담 및 업무 처리 | 상담 시간과 업무 처리 시간을 효율적으로 관리하여 고객 대기 시간 최소화 |',
}


vocs = [voc1, voc2, voc3, voc4, voc5]

In [38]:
"""
V4
+ 고객경험요소 추가
"""
from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Emotino Generate Index: {index} VOC:")
    print(f"{voc}")
    index = index + 1
    
    human_prompt = f"""
    ## 입력:
    
    **문항 질문**
    {voc['question']}
    
    ---
    
    **문항 응답내용**
    {voc['answer']}
    
    ---
    
    **VOC 원문**
    {voc['content']}
    
    ---
    
    **고객경험요소**
    | 고객경험요소 | 설명 |
    |--------------|------|
    {voc['cxe']}
    """.strip()

    messages = [
        SystemMessage(content=system_prompt_v4_detail),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Emotino Generate Index: 0 VOC:
{'content': '대기시간이 길지 않았다', 'question': 'Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?', 'answer': '고객 대기시간을 줄이기 위한 노력', 'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |'}
{
    "result": "만족"
}
Emotino Generate Index: 1 VOC:
{'content': '대기 시간이 길어요', 'question': 'Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?', 'answer': '확인하기 쉬운 대기정보(대기순번, 대기예상시간)', 'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |'}
{
    "result": "아쉬움"
}
Emotino Generate Index: 2 VOC:
{'content': 'ATM기 이용이 정말정말 편해요!!!', 'question': 'Q2-1.친구 또는 지인에게 ${부서명}지점의 영업점 내점/방문과정 경험을 적극적으로 추천하는 이유는 무엇입니까?', 'answer': '이용하기 쉬운 자동화기기(ATM, STM 등)', 'cxe': '| ATM 이용 안내의 명확성 | 장애인/외국인 등 약자를 위한 음성 안내, 다국어 지원 기능의 유무 및 품질 |'}
{
    "result": "감동"
}
Emotino Generate Index: 3 VOC:
{'content': '상담을통한 적절한상품과다양한상품에대한충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다', 'question': '개선의견 (주관식)', 'answer': '', 'cxe': '| 다양한 상

### 3. 결과 세팅
1. 응답 JSON 파싱
2. 결과 세팅

In [12]:
"""
import
"""
from langchain_core.output_parsers import JsonOutputParser

In [29]:
# Json Parse
# Parsing 실패 시, 예외 처리 -> OutputFixingParser 고민
parser = JsonOutputParser()
response = parser.parse(output)

print(response['emotion_score'])

result_dict = dict()
for key in feel_dict:

    emotion_score = response['emotion_score'].get(key) if response['emotion_score'].get(key) else 0.0
    emotion_info = feel_dict[key]
    # 생성된 최종 스코어가 포함된 결과 딕셔너리 세팅 
    emotion_result = {
        'name': emotion_info['name'],
        'score': emotion_score,
        'feel_large_cd': emotion_info['feel_large_cd'],
        'voc_type_cd': emotion_info['voc_type_cd']
    }
    result_dict[key] = emotion_result
    
print(result_dict)

{'04': 0.8, '06': 0.5, '02': 0.1}
{'01': {'name': '감동', 'score': 0.0, 'feel_large_cd': '01', 'voc_type_cd': '01'}, '02': {'name': '만족', 'score': 0.1, 'feel_large_cd': '01', 'voc_type_cd': '01'}, '03': {'name': '기대', 'score': 0.0, 'feel_large_cd': '01', 'voc_type_cd': '03'}, '04': {'name': '아쉬움', 'score': 0.8, 'feel_large_cd': '02', 'voc_type_cd': '03'}, '05': {'name': '실망', 'score': 0.0, 'feel_large_cd': '02', 'voc_type_cd': '02'}, '06': {'name': '짜증', 'score': 0.5, 'feel_large_cd': '02', 'voc_type_cd': '02'}, '07': {'name': '감정 없음', 'score': 0.0, 'feel_large_cd': '03', 'voc_type_cd': '04'}}


#### 최적의 감정 추출 방법론
1. 기 감정 데이터 활용 (Score 연산 후 비교)
    1. 감정 대분류를 기준으로 score 총합 구하기
    2. 기 데이터의 감정 대분류 별 score와 합산
        1. 최상위 대분류가 동일할 경우,
           - KB STA로 추출된 정답 감정(중분류) score가 임계치를 못 넘으면, LLM 중분류 채택
           - 임계치를 넘었으면, 기 정답 감정 유지
        2. 최상위 감정 대분류가 다를 경우,
           - LLM 중분류 채택
2. LLM만 활용
   - score로 분류할 것 없이, 가장 연관되었다고 판단되는 중분류 채택

In [58]:
# utils
def print_result(emotion_code: str, result_dict: dict) -> None:
    max_feel_info = result_dict[emotion_code]

    score = max_feel_info['score']
    mid = max_feel_info['name']
    large = feel_large_dict[max_feel_info['feel_large_cd']]
    voc_type = voc_type_dict[max_feel_info['voc_type_cd']]
    
    # =====================결과 출력================
    print(f'score >> {score}')
    print(f'mid >> {mid}')
    print(f'large >> {large}')
    print(f'voc_type >> {voc_type}')

In [59]:
"""
단순 추출
"""
max_feel_cd = max(result_dict, key=lambda key: result_dict[key]['score'])
print_result(max_feel_cd, result_dict)

score >> 0.8
mid >> 아쉬움
large >> 부정
voc_type >> 개선


In [74]:
"""
Regacy 결과 활용
"""
from collections import defaultdict


# 1. 감정 대분류 별로 grouping된 합산
feel_large_cd_sum = defaultdict(float)

for value in result_dict.values():
    feel_large_cd = value['feel_large_cd']
    score = value['score']
    feel_large_cd_sum[feel_large_cd] += score

print(dict(feel_large_cd_sum))

# 2. Regacy + LLM 대분류 점수 합산
regacy_large_sum = {
    '01': voc_info['affirmation_score'],
    '02': voc_info['negation_score'],
    '03': voc_info['neutrality_score']
}
print(regacy_large_sum)
final_large_sum = {
    code: dict(feel_large_cd_sum)[code] + regacy_large_sum[code]
    for code in feel_large_cd_sum.keys()
}

print(final_large_sum)

# 3. 최종 감정 대분류 선정
max_final_large_cd = max(final_large_sum, key=final_large_sum.get)
print(max_final_large_cd)

{'01': 0.1, '02': 1.3, '03': 0.0}
{'01': 0.9, '02': 0.3, '03': 0.0}
{'01': 1.0, '02': 1.6, '03': 0.0}
02


In [75]:
EMOTION_THRESHOLD = 0.5

if max_final_large_cd == voc_info['emotion_large_cd']:
    # 감정 대분류가 기존과 같을 때
    if voc_info['emotion_medium_score'] < EMOTION_THRESHOLD:
        # 임계치보다 작을 경우, 생성한 해당 대분류 중에서 가장 높은 중분류 채택
        filtered_data = {key: value for key, value in result_dict.items() if value['feel_large_cd'] == max_final_large_cd}
        final_emotion_medium_cd = max(filtered_data, key=lambda x: filtered_data[x]['score'])
    
    else:
        # 임계치보다 클 경우, Regacy 결과 유지
        final_emotion_medium_cd = voc_info['emotion_medium_cd']
    
else:
    # 감정 대분류가 기존과 다를 때
    # 새롭게 생성된 최종 감정 중분류를 채택
    final_emotion_medium_cd = max_feel_cd = max(result_dict, key=lambda key: result_dict[key]['score'])

print_result(final_emotion_medium_cd, result_dict)

score >> 0.8
mid >> 아쉬움
large >> 부정
voc_type >> 개선
